In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.svm import OneClassSVM
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
import matplotlib.pyplot as plt

# ============================================================
# 1. SETTINGS
# ============================================================

kernel = "rbf"
gamma = "auto"
nu = 0.05

threshold_list = [-0.5, -0.3,  0]

results_root = Path("../results/OCSVM_ECG5000_thresholds")
results_root.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. LOAD ECG5000 DATASET
# ============================================================

train_df = pd.read_csv(
    "../data_raw/ECG5000/ECG5000_TRAIN.txt",
    sep=r"\s+",
    header=None
)

test_df = pd.read_csv(
    "../data_raw/ECG5000/ECG5000_TEST.txt",
    sep=r"\s+",
    header=None
)

y_train_orig = train_df.iloc[:, 0].values
X_train_all = train_df.iloc[:, 1:].values

y_test_orig = test_df.iloc[:, 0].values
X_test = test_df.iloc[:, 1:].values

# ============================================================
# 3. DEFINE NORMAL / ANOMALY CLASSES
# ============================================================

normal_class = 1

# 1 = normal, -1 = anomaly
y_test = np.where(y_test_orig == normal_class, 1, -1)

# ============================================================
# 4. TRAIN ONLY ON NORMAL DATA
# ============================================================

X_train_normal = X_train_all[y_train_orig == normal_class]

# ============================================================
# 5. STANDARDIZATION
# ============================================================

mu = X_train_normal.mean(axis=0)
sigma = X_train_normal.std(axis=0)
sigma[sigma == 0] = 1

X_train_normal_z = (X_train_normal - mu) / sigma
X_test_z = (X_test - mu) / sigma

# ============================================================
# 6. TRAIN FIXED ONE-CLASS SVM
# ============================================================

model = OneClassSVM(
    kernel=kernel,
    gamma=gamma,
    nu=nu
)

model.fit(X_train_normal_z)

scores = model.decision_function(X_test_z).flatten()
score_samples = model.score_samples(X_test_z).flatten()
pred_default = model.predict(X_test_z)

print("Decision score min:", scores.min())
print("Decision score max:", scores.max())
print("Decision score mean:", scores.mean())

# ============================================================
# 7. EXPERIMENTS WITH THRESHOLDS
# ============================================================

all_results = []

for threshold in threshold_list:
    exp_name = f"kernel_{kernel}_nu_{nu}_gamma_{gamma}_threshold_{threshold}"
    exp_name = exp_name.replace(".", "_").replace("-", "minus_")

    exp_dir = results_root / exp_name
    exp_dir.mkdir(parents=True, exist_ok=True)

    pred_custom = np.ones(len(scores))
    pred_custom[scores < threshold] = -1

    accuracy = accuracy_score(y_test, pred_custom) * 100
    precision = precision_score(y_test, pred_custom, pos_label=-1, zero_division=0) * 100
    recall = recall_score(y_test, pred_custom, pos_label=-1, zero_division=0) * 100
    f1 = f1_score(y_test, pred_custom, pos_label=-1, zero_division=0) * 100

    cm = confusion_matrix(y_test, pred_custom, labels=[-1, 1])

    result_row = {
        "experiment": exp_name,
        "kernel": kernel,
        "nu": nu,
        "gamma": gamma,
        "threshold": threshold,
        "train_normal_samples": len(X_train_normal_z),
        "test_samples": len(X_test_z),
        "test_normal_samples": int((y_test == 1).sum()),
        "test_anomaly_samples": int((y_test == -1).sum()),
        "accuracy_%": round(accuracy, 2),
        "precision_%": round(precision, 2),
        "recall_%": round(recall, 2),
        "f1_%": round(f1, 2)
    }

    all_results.append(result_row)

    predictions_df = pd.DataFrame({
        "y_true": y_test,
        "pred_default": pred_default,
        "pred_custom": pred_custom.astype(int),
        "decision_score": scores,
        "score_samples": score_samples,
        "threshold": threshold
    })

    predictions_df.to_csv(exp_dir / "predictions.csv", index=False)

    print("======================================")
    print("One-Class SVM ECG5000")
    print("======================================")
    print(f"Threshold: {threshold}")
    print(f"Accuracy : {accuracy:.2f}%")
    print(f"Precision: {precision:.2f}%")
    print(f"Recall   : {recall:.2f}%")
    print(f"F1-score : {f1:.2f}%")
    print("\nConfusion Matrix")
    print("Rows = true labels [-1 anomaly, 1 normal]")
    print("Cols = predicted labels [-1 anomaly, 1 normal]")
    print(cm)

    # ========================================================
    # Metrics table
    # ========================================================

    metrics_df = pd.DataFrame({
        "Metrika": [
            "Presnosť (Accuracy)",
            "Precíznosť (Precision)",
            "Citlivosť (Recall)",
            "F1-skóre"
        ],
        "Hodnota (%)": [
            round(accuracy, 2),
            round(precision, 2),
            round(recall, 2),
            round(f1, 2)
        ]
    })

    fig, ax = plt.subplots(figsize=(6, 2))
    ax.axis("off")

    table = ax.table(
        cellText=metrics_df.values,
        colLabels=metrics_df.columns,
        loc="center",
        cellLoc="center"
    )

    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1, 2)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold")
            cell.set_facecolor("#dce6f1")
        else:
            cell.set_facecolor("#f9f9f9")

    plt.title(
        f"Výsledné metriky One-Class SVM – ECG5000\nThreshold = {threshold}",
        fontsize=12,
        pad=20
    )

    plt.savefig(exp_dir / "metrics_table.png", dpi=300, bbox_inches="tight")
    plt.close()

    # ========================================================
    # Confusion matrix
    # ========================================================

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"Matica zámen – Threshold = {threshold}")
    plt.colorbar()

    classes = ["Anomálie", "Normálne"]
    tick_marks = np.arange(len(classes))

    plt.xticks(tick_marks, classes)
    plt.yticks(tick_marks, classes)

    threshold_cm = cm.max() / 2.0 if cm.max() > 0 else 0.5

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = "black" if cm[i, j] > threshold_cm else "white"
            plt.text(
                j, i, cm[i, j],
                ha="center",
                va="center",
                color=color
            )

    plt.xlabel("Predikované triedy")
    plt.ylabel("Skutočné triedy")
    plt.tight_layout()
    plt.savefig(exp_dir / "confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.close()

    # ========================================================
    # Score histogram
    # ========================================================

    plt.figure(figsize=(10, 5))

    plt.hist(
        scores[y_test == 1],
        bins=50,
        alpha=0.7,
        label="Normálne"
    )

    plt.hist(
        scores[y_test == -1],
        bins=50,
        alpha=0.7,
        label="Anomálie"
    )

    plt.axvline(
        threshold,
        linestyle="--",
        linewidth=2,
        label=f"Threshold = {threshold}"
    )

    plt.title(f"Histogram skóre One-Class SVM – ECG5000\nThreshold = {threshold}")
    plt.xlabel("Decision score")
    plt.ylabel("Počet vzoriek")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(exp_dir / "score_histogram.png", dpi=300, bbox_inches="tight")
    plt.close()

    # ========================================================
    # Example ECG signals
    # ========================================================

    normal_idx = np.where(y_test == 1)[0][0]
    anom_idx = np.where(y_test == -1)[0][0]

    plt.figure(figsize=(10, 6))

    plt.subplot(2, 1, 1)
    plt.plot(X_test[normal_idx])
    plt.title("Normálny EKG signál")
    plt.grid(True)

    plt.subplot(2, 1, 2)
    plt.plot(X_test[anom_idx])
    plt.title("Anomálny EKG signál")
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(exp_dir / "ecg_examples.png", dpi=300, bbox_inches="tight")
    plt.close()

# ============================================================
# 8. SAVE SUMMARY
# ============================================================

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(by="f1_%", ascending=False).reset_index(drop=True)
results_df.to_csv(results_root / "summary_results.csv", index=False)

print("Done")

Decision score min: -0.801259508858507
Decision score max: 0.3919077695578037
Decision score mean: -0.1714299912400494
One-Class SVM ECG5000
Threshold: -0.5
Accuracy : 92.64%
Precision: 96.78%
Recall   : 85.16%
F1-score : 90.60%

Confusion Matrix
Rows = true labels [-1 anomaly, 1 normal]
Cols = predicted labels [-1 anomaly, 1 normal]
[[1595  278]
 [  53 2574]]
One-Class SVM ECG5000
Threshold: -0.3
Accuracy : 96.87%
Precision: 94.59%
Recall   : 98.08%
F1-score : 96.30%

Confusion Matrix
Rows = true labels [-1 anomaly, 1 normal]
Cols = predicted labels [-1 anomaly, 1 normal]
[[1837   36]
 [ 105 2522]]
One-Class SVM ECG5000
Threshold: 0
Accuracy : 91.53%
Precision: 83.21%
Recall   : 99.79%
F1-score : 90.75%

Confusion Matrix
Rows = true labels [-1 anomaly, 1 normal]
Cols = predicted labels [-1 anomaly, 1 normal]
[[1869    4]
 [ 377 2250]]
Done
